In [ ]:
import io
import zipfile

import frontmatter
import requests


In [34]:
def read_github_repo(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith((".md", ".mdx")):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode("utf_8")
                post = frontmatter.loads(content)
                doc = {
                    "content": post.content,
                    "title": post.metadata.get("title"),
                    "description": post.metadata.get("description"),
                    "filename": file_path.split("/", 1)[-1],
                }
                documents.append(doc)
    return documents


In [35]:
repo_owner = "evidentlyai"
repo_name = "docs"

documents = read_github_repo(repo_owner, repo_name)
print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [47]:
# using gitsource lib
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")
documents = [f.parse() for f in files]


Loaded 95 documents
